# Filter Dataset Penyakit THT

Notebook ini digunakan untuk memisahkan dataset `Diseases_and_Symptoms_dataset.csv` agar hanya berfokus pada 5 penyakit THT yang dipilih pada projek akhir:

1. Otitis Media
2. Ear Drum Damage
3. Eustachian Tube Dysfunction
4. Nose Disorder
5. Obstructive Sleep Apnea

Dataset hasil filter akan disimpan ke `dataset/penyakit_tht_5.csv`.

## 1. Import Library

In [ ]:
import csv
from pathlib import Path

import pandas as pd

## 2. Tentukan Lokasi Dataset

In [ ]:
DATASET_PATH = Path("dataset/Diseases_and_Symptoms_dataset.csv")
OUTPUT_PATH = Path("dataset/penyakit_tht_5.csv")

DATASET_PATH.exists()

## 3. Baca Dataset

Header pada file CSV ini dibungkus satu tanda kutip besar, sehingga perlu diperbaiki terlebih dahulu sebelum dibaca sebagai 231 kolom.

In [ ]:
def read_diseases_dataset(path: Path) -> pd.DataFrame:
    with path.open("r", encoding="utf-8", newline="") as file:
        header_line = file.readline().rstrip("\r\n")

    if header_line.startswith('"') and header_line.endswith('"'):
        header_line = header_line[1:-1].replace('""', '"')

    columns = next(csv.reader([header_line]))
    return pd.read_csv(path, header=None, names=columns, skiprows=1)


df = read_diseases_dataset(DATASET_PATH)
df.head()

## 4. Informasi Dataset Awal

In [ ]:
print(f"Jumlah baris dataset awal: {df.shape[0]}")
print(f"Jumlah kolom dataset awal: {df.shape[1]}")
print(f"Jumlah penyakit unik: {df['diseases'].nunique()}")

In [ ]:
df["diseases"].value_counts().sort_index().head(20)

## 5. Tentukan 5 Penyakit THT

Dua nama penyakit pada dataset memiliki label lebih lengkap dibandingkan yang tertulis di dokumen projek:

- `eustachian tube dysfunction` ditulis sebagai `eustachian tube dysfunction (ear disorder)`
- `obstructive sleep apnea` ditulis sebagai `obstructive sleep apnea (osa)`

In [ ]:
target_diseases = [
    "otitis media",
    "ear drum damage",
    "eustachian tube dysfunction (ear disorder)",
    "nose disorder",
    "obstructive sleep apnea (osa)",
]

target_diseases

## 6. Filter Dataset Hanya Untuk Penyakit THT

In [ ]:
df_tht = df[df["diseases"].isin(target_diseases)].copy()

print(f"Jumlah baris setelah filter: {df_tht.shape[0]}")
print(f"Jumlah kolom setelah filter: {df_tht.shape[1]}")
print(f"Jumlah penyakit THT: {df_tht['diseases'].nunique()}")

df_tht.head()

## 7. Validasi Hasil Filter

In [ ]:
disease_counts = df_tht["diseases"].value_counts().reindex(target_diseases)
disease_counts

In [ ]:
unexpected_diseases = sorted(set(df_tht["diseases"].unique()) - set(target_diseases))
missing_diseases = sorted(set(target_diseases) - set(df_tht["diseases"].unique()))

print("Penyakit di luar target:", unexpected_diseases)
print("Penyakit target yang tidak ditemukan:", missing_diseases)

## 8. EDA Gejala Dominan Tiap Penyakit

Bagian ini menampilkan gejala yang paling sering muncul untuk masing-masing penyakit THT. Nilai `frekuensi` menunjukkan proporsi baris penyakit tersebut yang memiliki gejala bernilai `1`.

In [ ]:
symptom_columns = [column for column in df_tht.columns if column != "diseases"]

def top_symptoms_by_disease(data: pd.DataFrame, disease: str, top_n: int = 10) -> pd.DataFrame:
    disease_rows = data[data["diseases"] == disease]
    symptom_frequency = disease_rows[symptom_columns].mean().sort_values(ascending=False)
    symptom_frequency = symptom_frequency[symptom_frequency > 0].head(top_n)

    return symptom_frequency.reset_index().rename(
        columns={"index": "gejala", 0: "frekuensi"}
    )


top_symptom_tables = {
    disease: top_symptoms_by_disease(df_tht, disease, top_n=10)
    for disease in target_diseases
}

top_symptom_tables[target_diseases[0]]

In [ ]:
for disease, table in top_symptom_tables.items():
    print(f"\n=== {disease} ===")
    display(table)

## 9. Simpan Dataset Hasil Filter

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_tht.to_csv(OUTPUT_PATH, index=False)

print(f"Dataset THT berhasil disimpan ke: {OUTPUT_PATH}")

## 10. Cek Ulang File Output

In [ ]:
df_check = pd.read_csv(OUTPUT_PATH)

print(f"Jumlah baris file output: {df_check.shape[0]}")
print(f"Jumlah kolom file output: {df_check.shape[1]}")
print("Label penyakit pada file output:")
display(df_check["diseases"].value_counts().reindex(target_diseases))